In [0]:
spark

In [0]:
# Import Required Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Import Required Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Read Dataset
reviews_df = spark.read \
    .format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .load("/Volumes/workspace/default/train/")

In [0]:
reviews_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)



In [0]:
# Assign Meaningful Column Names
reviews_df = reviews_df.toDF(
    "label",
    "title",
    "review"
)

In [0]:
#Data Exploration
#4.1 Print Column Names
print(reviews_df.columns)

['_c0', '_c1', '_c2']


In [0]:
# Count Total Rows
print("Total Rows :", reviews_df.count())

Total Rows : 3600000


In [0]:
# Display Schema
reviews_df.printSchema()

root
 |-- label: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- review: string (nullable = true)



In [0]:
# Count Total Columns
print("Total Columns :", len(reviews_df.columns))

Total Columns : 3


In [0]:
# Display Sample Records
reviews_df.show(10, truncate=False)

+-----+------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|title                                                 |review                                                                    

In [0]:
# Corrupted Record Detection
if "_corrupt_record" in reviews_df.columns:
    reviews_df.filter(col("_corrupt_record").isNotNull()).show()
else:
    print("No Corrupted Records Found")

No Corrupted Records Found


In [0]:

# Create Custom Schema
from pyspark.sql.types import *
schema = StructType([
    StructField("label", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("review", StringType(), True)
])

reviews_df = spark.read \
    .format("csv") \
    .schema(schema) \
    .option("header", "false") \
    .load("/Volumes/workspace/default/amazon_reviews/train.csv")

In [0]:
reviews_df.printSchema()

root
 |-- label: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- review: string (nullable = true)



In [0]:
# Transformation 1 : Alias Columns
from pyspark.sql.functions import col
reviews_df.select(
    col("title").alias("Review_Title"),
    col("review").alias("Review_Text")
).show(5)

+--------------------+--------------------+
|        Review_Title|         Review_Text|
+--------------------+--------------------+
|Stuning even for ...|This sound track ...|
|The best soundtra...|I'm reading a lot...|
|            Amazing!|"This soundtrack ...|
|Excellent Soundtrack|I truly like this...|
|Remember, Pull Yo...|If you've played ...|
+--------------------+--------------------+
only showing top 5 rows


In [0]:
# Transformation 2 : Filter Positive Reviews
reviews_df.filter(col("label")==2).show(5)

+-----+--------------------+--------------------+
|label|               title|              review|
+-----+--------------------+--------------------+
|    2|Stuning even for ...|This sound track ...|
|    2|The best soundtra...|I'm reading a lot...|
|    2|            Amazing!|"This soundtrack ...|
|    2|Excellent Soundtrack|I truly like this...|
|    2|Remember, Pull Yo...|If you've played ...|
+-----+--------------------+--------------------+
only showing top 5 rows


In [0]:
# Transformation 3 : Add Constant Column
reviews_df = reviews_df.withColumn(
    "country",
    lit("USA")
)

In [0]:
# Transformation 4 : Calculate Review Length
reviews_df = reviews_df.withColumn(
    "review_length",
    length(col("review"))
)

In [0]:
# Transformation 5 : Rename Column
reviews_df = reviews_df.withColumnRenamed(
    "label",
    "sentiment"
)

In [0]:
# Transformation 6 : Cast Data Type
reviews_df = reviews_df.withColumn(
    "sentiment",
    col("sentiment").cast("int")
)

In [0]:
# Transformation 7 : Drop Column
reviews_df = reviews_df.drop("country")

In [0]:
# Count Null Values
reviews_df.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in reviews_df.columns
]).show()

+---+---+---+
|_c0|_c1|_c2|
+---+---+---+
|  0| 48| 13|
+---+---+---+



In [0]:
reviews_df = reviews_df.na.fill({
    "_c1": "Unknown",
    "_c2": "No Review"
})

In [0]:
from pyspark.sql.functions import col, sum

reviews_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in reviews_df.columns]
).show()

+---+---+---+
|_c0|_c1|_c2|
+---+---+---+
|  0|  0|  0|
+---+---+---+



In [0]:
# Remove Duplicate Records
reviews_df = reviews_df.dropDuplicates()

In [0]:
# Final Data Preview
reviews_df.show(10,truncate=False)

+---------+-------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+
|sentiment|title                                      |review                             

In [0]:
# Save Processed Dataset
reviews_df.write \
.mode("overwrite") \
.parquet("/Volumes/workspace/default/task/amazon_reviews")

In [0]:
reviews_df.show(10, truncate=False)

+---+------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_c0|_c1                                                   |_c2                                                                           

In [0]:
spark.sql("SHOW VOLUMES").show(truncate=False)

+--------+-----------+
|database|volume_name|
+--------+-----------+
|default |abc        |
|default |airports   |
|default |corrupted  |
|default |def        |
|default |dfe        |
|default |flight     |
|default |flight_data|
|default |fllightdata|
|default |kaggle     |
|default |loading    |
|default |movies     |
|default |multi      |
|default |task       |
|default |train      |
|default |zomato     |
+--------+-----------+



In [0]:
# Read Saved Parquet File
final_df = spark.read.parquet(
    "/Volumes/workspace/default/train/processed_reviews"
)

final_df.show(5)

+---------+--------------------+--------------------+-------------+
|sentiment|               title|              review|review_length|
+---------+--------------------+--------------------+-------------+
|        2|Hanford Mills museum|My friend is a ma...|          188|
|        2|One of the very b...|"This is one of t...|           78|
|        2|            Good DVD|Enjoyed it immens...|          132|
|        1|No Editor? Medioc...|After watching a ...|          637|
|        1|This book is awfu...|"Even for the fre...|          148|
+---------+--------------------+--------------------+-------------+
only showing top 5 rows
